## Qutip Tutorial

#### Imports

In [1]:
import qutip
from qutip import operators
from typing import List,Tuple
import numpy as np

#### First: Define the transverse ising model 

In qutip the spin basis is simply a Fock representation with 2 D.O.F (Hardcore bosons). If we want to write the hamiltonian we should consider the tensor product operator for each term (it is not like quspin).

#### Define the general operator method

We implement a function in order to compute the general many-body operator

$O^{a_1,a_2,...,a_n}_{i_1, i_2, ..., i_n}=G^{a_1, a_2, .. a_n}_{i_1,i_2,...,i_n} s^{a_1}_{i_1} \otimes s^{a_2}_{i_2} \otimes ... \otimes s^{a_n}_{i_n}$

In [2]:
pauli={'idx':qutip.identity(2),'x':qutip.sigmax(),'y':qutip.sigmay(),'z':qutip.sigmaz()}

def manybodyoperator(directions:List[Tuple[str,List]],size:int)->qutip.Qobj:

    # for each coupling term in the list direction
    for r,(coupling,direction) in enumerate(directions):
        # starting point -> identity operator
        idx_mb:List[str]=['idx' for i in range(size)]
        
        # create the op representation
        for dir,i in direction:
            #print('dir=',dir,'i=',i)
            idx_mb[i]=dir
        # convert into qutip.Qobj
        for i in range(size):
            if i==0:
                op=pauli[idx_mb[i]]        
            else:
                op=qutip.tensor(op,pauli[idx_mb[i]])
        #sum each direction
        if r==0:
            manybodyop=op*coupling
        else:
            manybodyop=manybodyop+op*coupling
                
    return manybodyop


#### Test the Method

We study the operator $O_{i ,i+1}=\sum^{N-1}_i x_i \otimes x_{i+1}$

In [3]:
size:int=3
coupling:float=1
directions:List[Tuple[float,List]]=[(coupling,[('x',i),('x',i+1)]) for i in range(size-1)]
op:qutip.Qobj=manybodyoperator(directions=directions,size=size)

print(op)

Quantum object: dims = [[2, 2, 2], [2, 2, 2]], shape = (8, 8), type = oper, isherm = True
Qobj data =
[[0. 0. 0. 1. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0. 0. 1. 0.]
 [1. 0. 0. 0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 1. 0. 0. 0.]]


#### Implement a SteadyState class

In [12]:

class SteadyStateClass():
    
    def __init__(self,size:int,unitary_list:List[List],dissipative_list:List[List]) -> None:
        
        #parameters
        self.unitary_list:List[List]=unitary_list
        self.dissipative_list:List[List]=dissipative_list
        self.size=size
        
        #attributes
        self.steady_state:qutip.Qobj=None
        self.limbladian:qutip.Qobj=None
    
    
    #operation that convert the abstract string to the qutip.Qobj    
    def _manybodyoperator(self,directions:List[Tuple[str,List]],size:int)->qutip.Qobj:

        # for each coupling term in the list direction
        for r,(coupling,direction) in enumerate(directions):
            # starting point -> identity operator
            idx_mb:List[str]=['idx' for i in range(size)]
            
            # create the op representation
            for dir,i in direction:
                #print('dir=',dir,'i=',i)
                idx_mb[i]=dir
            # convert into qutip.Qobj
            for i in range(size):
                if i==0:
                    op=pauli[idx_mb[i]]        
                else:
                    op=qutip.tensor(op,pauli[idx_mb[i]])
            #sum each direction
            if r==0:
                manybodyop=op*coupling
            else:
                manybodyop=manybodyop+op*coupling
                    
        return manybodyop
    
    def _get_the_limbladian(self)->None:
        #define the hamiltonian
        for i,u in enumerate(self.unitary_list):
            if i==0:
                hamiltonian=self._manybodyoperator(directions=u,size=self.size)
            else:
                hamiltonian=hamiltonian+self._manybodyoperator(directions=u,size=self.size)
        dissipative=[]
        for d in self.dissipative_list:
            dissipative.append(self._manybodyoperator(directions=d,size=self.size))
        self.limbladian=qutip.liouvillian(H=hamiltonian,c_ops=dissipative)
        
    def get_steady_state(self)->qutip.Qobj:
        self._get_the_limbladian()
        self.steady_state=qutip.steadystate(qutip.to_super(self.limbladian))

    
    def print_liouvillian(self)->None:
        print('Unitary part=\n',self.unitary_list,'\n')
        print('Dissipative part=\n',self.dissipative_list,'\n')
    

#### Check if the SteadyState Class works

In [13]:
size:int=3
j:float=1
h:float=1
g:float=0.1
xx:List[Tuple[float,List]]=[(j,[('x',i),('x',i+1)]) for i in range(size-1)]
z:List[Tuple[float,List]]=[(h,[('z',i)]) for i in range(size)]
x:List[Tuple[float,List]]=[(g,[('x',i)]) for i in range(size)]

unitary=[xx,z]
dissipative=[x]

std=SteadyStateClass(size=size,unitary_list=unitary,dissipative_list=dissipative)

std.print_liouvillian()

std.get_steady_state()

print(std.steady_state)



Unitary part=
 [[(1, [('x', 0), ('x', 1)]), (1, [('x', 1), ('x', 2)])], [(1, [('z', 0)]), (1, [('z', 1)]), (1, [('z', 2)])]] 

Dissipative part=
 [[(0.1, [('x', 0)]), (0.1, [('x', 1)]), (0.1, [('x', 2)])]] 

Quantum object: dims = [[2, 2, 2], [2, 2, 2]], shape = (8, 8), type = oper, isherm = True
Qobj data =
[[ 1.25157600e-01+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  1.01176148e-17+4.01947075e-20j
   0.00000000e+00+0.00000000e+00j -7.27368289e-18+1.98309356e-20j
   1.36060622e-17+9.70710029e-21j  0.00000000e+00+0.00000000e+00j]
 [ 0.00000000e+00+0.00000000e+00j  1.24842400e-01+0.00000000e+00j
   2.40510938e-17-2.25901977e-19j  0.00000000e+00+0.00000000e+00j
   3.15200492e-04-1.35525272e-19j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  2.43657802e-18+3.68656875e-21j]
 [ 0.00000000e+00+0.00000000e+00j  2.40510938e-17+2.25901977e-19j
   1.25157600e-01+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   2.32033782e-17-2.64063895